In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 22. DPOテキスト テキスト テキスト テキスト — RLHF テキスト テキスト

> **学習目標**
> - DPOテキスト RLHFテキスト 3テキスト 1テキスト テキスト テキスト テキスト度テキスト
> - Bradley-Terry モデルテキスト DPO テキスト テキスト テキスト
> - テキスト テキスト テキスト(KTO, IPO, ORPO)テキスト テキスト テキスト

## 22.1 RLHFテキスト テキスト DPOテキスト テキスト

RLHFテキスト 問題:
- 4テキスト モデル (policy, reference, RM, value) → テキスト テキスト
- PPO テキスト テキスト, 学習 テキスト
- RM テキスト 学習 → テキスト テキスト

**DPO** (Direct Preference Optimization, Rafailov et al., 2023):
- RM 学習 + PPOテキスト **テキスト テキスト**テキスト テキスト
- テキスト データテキスト テキスト 学習
- テキスト: 2テキスト モデルテキスト (policy, reference)


In [ ]:

# テキスト モデル テキスト (Ch 18テキスト テキスト)
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def attention(self, x, mask):
        B, T, D = x.shape
        q, k, v = self.W_qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5) + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out)

    def forward(self, x, mask):
        x = x + self.dropout(self.attention(self.ln1(x), mask))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=2,
                 d_ff=256, max_seq_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight  # weight tying
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.token_emb(x) * (self.d_model ** 0.5) + self.pos_emb(positions)
        h = self.dropout(emb)
        mask = torch.triu(torch.full((T, T), float('-inf'), device=x.device), diagonal=1)
        for block in self.blocks:
            h = block(h, mask)
        h = self.ln_f(h)
        return self.lm_head(h)

print("MiniGPT Model テキスト テキスト")


In [ ]:
# テキスト テキスト テキスト (Ch 22 テキスト)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

vocab_size = 100  # toy
print("vocab_size (toy):", vocab_size)


## 22.2 Bradley-Terry テキスト モデル テキスト

$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma(r(\mathbf{x}, \mathbf{y}_w) - r(\mathbf{x}, \mathbf{y}_l))$$

RM テキスト: $\mathcal{L}_{\text{RM}} = -\log \sigma(r_w - r_l)$

## 22.3 テキスト-テキスト テキスト テキスト度

KL テキスト テキスト 問題:
$$\max_\pi \mathbb{E}_{\mathbf{y} \sim \pi}[r(\mathbf{x}, \mathbf{y})] - \beta D_{\text{KL}}(\pi(\cdot|\mathbf{x}) \| \pi_{\text{ref}}(\cdot|\mathbf{x}))$$

テキスト テキスト テキスト (テキスト 解法):
$$\pi^*(\mathbf{y}|\mathbf{x}) = \frac{\pi_{\text{ref}}(\mathbf{y}|\mathbf{x}) \exp(r(\mathbf{x}, \mathbf{y}) / \beta)}{Z(\mathbf{x})}$$

テキスト:
$$r(\mathbf{x}, \mathbf{y}) = \beta \log \frac{\pi^*(\mathbf{y}|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}|\mathbf{x})} + \beta \log Z(\mathbf{x})$$

$\log Z(\mathbf{x})$テキスト $\mathbf{y}$テキスト テキスト, $\mathbf{y}_w$ vs $\mathbf{y}_l$ 比較テキスト テキスト.

## 22.4 DPO テキスト テキスト度

テキスト-テキスト テキスト Bradley-Terryテキスト テキスト:

$$P(\mathbf{y}_w \succ \mathbf{y}_l | \mathbf{x}) = \sigma\left(\beta \log \frac{\pi_\theta(\mathbf{y}_w|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_w|\mathbf{x})} - \beta \log \frac{\pi_\theta(\mathbf{y}_l|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_l|\mathbf{x})}\right)$$

**DPO テキスト**:
$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(\mathbf{y}_w|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_w|\mathbf{x})} - \beta \log \frac{\pi_\theta(\mathbf{y}_l|\mathbf{x})}{\pi_{\text{ref}}(\mathbf{y}_l|\mathbf{x})}\right)\right]$$

テキスト: **RM テキスト** テキスト $\pi_\theta$テキスト テキスト 学習. テキスト テキスト テキスト テキスト.


In [ ]:
# DPO テキスト テキスト
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def get_seq_logprob(model, x, y_mask=None):
    """テキスト テキスト Probability Sum."""
    logits = model(x)
    log_probs = F.log_softmax(logits, dim=-1)
    # shift
    log_probs = log_probs[:, :-1, :]
    targets = x[:, 1:]
    # gather
    seq_log_probs = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)  # (B, T-1)
    if y_mask is not None:
        seq_log_probs = seq_log_probs * y_mask[:, 1:]
    return seq_log_probs.sum(dim=-1)  # (B,)

def dpo_loss(policy_chosen_logp, policy_rejected_logp,
             ref_chosen_logp, ref_rejected_logp, beta=0.1):
    """DPO Loss."""
    chosen_ratio = policy_chosen_logp - ref_chosen_logp
    rejected_ratio = policy_rejected_logp - ref_rejected_logp
    logits = beta * (chosen_ratio - rejected_ratio)
    return -F.logsigmoid(logits).mean()

# テキスト
# テキスト テキスト Probability (テキスト Modelテキスト Computation)
torch.manual_seed(0)
B = 8
ref_chosen_logp = torch.randn(B) - 5  # テキスト Modelテキスト chosen テキスト Probability
ref_rejected_logp = torch.randn(B) - 7  # rejectedテキスト テキスト テキスト テキスト

# policy テキスト: refテキスト テキスト
policy_chosen_logp = ref_chosen_logp.clone() + torch.randn(B) * 0.1
policy_rejected_logp = ref_rejected_logp.clone() + torch.randn(B) * 0.1

# Training テキスト (policyテキスト chosen テキスト)
losses = []
for step in range(100):
    # policyテキスト Pointテキスト chosen テキスト (gradient テキスト)
    policy_chosen_logp = policy_chosen_logp + 0.05
    # gradient ascent on chosen ratio

    loss = dpo_loss(policy_chosen_logp, policy_rejected_logp,
                    ref_chosen_logp, ref_rejected_logp, beta=0.1)
    losses.append(loss.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('DPO Loss')
plt.title('DPO Training テキスト')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch22_dpo_loss.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"テキスト loss: {losses[0]:.4f}, テキスト loss: {losses[-1]:.4f}")
print(f"chosen/rejected log-prob Difference: {(policy_chosen_logp - policy_rejected_logp).mean():.4f}")


## 22.5 DPO 学習ループ

テキスト テキスト:
1. SFT モデルテキスト $\pi_{\text{ref}}$テキスト テキスト
2. テキスト データ $(\mathbf{x}, \mathbf{y}_w, \mathbf{y}_l)$ テキスト
3. $\pi_\theta$テキスト $\pi_{\text{ref}}$ テキスト $\mathbf{y}_w, \mathbf{y}_l$テキスト テキスト テキスト 計算
4. DPO テキスト 計算, $\pi_\theta$テキスト バックプロパゲーション


In [ ]:
# DPO 学習 (テキスト)
import torch
import torch.nn as nn
import torch.nn.functional as F

# テキスト テキスト データ テキスト
def make_pref_batch(vocab_size, batch_size=4, seq_len=32):
    """chosenテキスト rejected テキスト Generation."""
    chosen = torch.randint(0, vocab_size, (batch_size, seq_len))
    # rejectedテキスト chosenテキスト テキスト テキスト テキスト
    rejected = chosen.clone()
    n_change = seq_len // 4
    for i in range(batch_size):
        idxs = torch.randperm(seq_len)[:n_change]
        rejected[i, idxs] = torch.randint(0, vocab_size, (n_change,))
    return chosen, rejected

# Model 2テキスト (policy, reference)
torch.manual_seed(0)
policy = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=64)
ref = MiniGPT(vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, max_seq_len=64)
ref.load_state_dict(policy.state_dict())  # テキスト テキスト
for p in ref.parameters():
    p.requires_grad = False  # refテキスト テキスト

opt = torch.optim.AdamW(policy.parameters(), lr=5e-4, weight_decay=0.01)

losses = []
for step in range(50):
    chosen, rejected = make_pref_batch(vocab_size, batch_size=4, seq_len=32)
    # テキスト Probability (テキスト テキスト)
    with torch.no_grad():
        ref_chosen_logp = get_seq_logprob(ref, chosen)
        ref_rejected_logp = get_seq_logprob(ref, rejected)
    policy_chosen_logp = get_seq_logprob(policy, chosen)
    policy_rejected_logp = get_seq_logprob(policy, rejected)

    loss = dpo_loss(policy_chosen_logp, policy_rejected_logp,
                    ref_chosen_logp, ref_rejected_logp, beta=0.1)
    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

plt.figure(figsize=(9, 3))
plt.plot(losses)
plt.xlabel('Step'); plt.ylabel('DPO Loss')
plt.title('DPO Training (MiniGPT)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch22_dpo_training.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"テキスト DPO loss: {losses[-1]:.4f}")


## 22.6 IPO, KTO, ORPO — DPOテキスト テキスト

### IPO (Identity Preference Optimization)
DPOテキスト テキスト 問題 テキスト:
$$\mathcal{L}_{\text{IPO}} = (\log \sigma(\ldots) - \frac{1}{2})^2$$

### KTO (Kahneman-Tversky Optimization)
**テキスト テキスト テキスト テキスト**. テキスト/テキスト テキスト テキスト テキスト テキスト テキスト:
$$\mathcal{L}_{\text{KTO}} = \sigma(\beta \log \frac{\pi}{\pi_{\text{ref}}} - z) + \lambda \sigma(z - \beta \log \frac{\pi}{\pi_{\text{ref}}})$$

データ テキスト テキスト テキスト (テキスト テキスト テキスト).

### ORPO (Odds Ratio Preference Optimization)
SFTテキスト DPOテキスト テキスト — reference modelテキスト テキスト テキスト. テキスト テキスト.


## 22.7 DPO vs PPO 比較

| テキスト | PPO (RLHF) | DPO |
|---|---|---|
| モデル テキスト | 4 (policy, ref, RM, value) | 2 (policy, ref) |
| 学習 テキスト | テキスト | テキスト |
| テキスト | テキスト | テキスト (β テキスト度) |
| テキスト | テキスト | テキスト |
| テキスト テキスト度 | テキスト | テキスト |
| テキスト | テキスト テキスト テキスト テキスト テキスト | テキスト |

テキスト DPOテキスト テキスト テキスト. テキスト モデル(Mistral, LLaMA-3 テキスト)度 DPO テキスト.


## 22.8 要点

| テキスト | テキスト テキスト | データ |
|---|---|---|
| RLHF | RM + PPO + KL | テキスト テキスト |
| DPO | テキスト-テキスト テキスト テキスト テキスト | テキスト テキスト |
| IPO | DPO + テキスト | テキスト テキスト |
| KTO | テキスト/テキスト テキスト | テキスト テキスト |
| ORPO | SFT+DPO テキスト | テキスト テキスト |

## 演習問題

1. DPO テキスト $\beta = 0.01, 0.1, 1.0$テキスト 比較テキスト テキスト テキスト.
2. Bradley-Terry モデルテキスト "テキスト テキスト"テキスト テキスト モデルテキスト テキスト.
3. DPOテキスト PPOテキスト テキスト テキスト 比較テキスト.
4. KTOテキスト テキスト テキスト テキスト テキスト 学習 テキスト テキスト.
5. テキスト-テキスト テキスト $r = \beta \log \frac{\pi^*}{\pi_{\text{ref}}} + \beta \log Z$テキスト テキスト度テキスト.

> 解答: `solutions/ch22_solutions.ipynb`
